# Perch v2 ONNX Speed Test

ONNX 版 Perch v2 の推論速度を計測し、全 train_soundscapes の埋め込み計算が
Kaggle GPU 1セッション（12時間）で完了するか見積もる。

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu', 'huggingface_hub'])

In [ ]:
# Cell 2 — Download ONNX model from HuggingFace
from huggingface_hub import hf_hub_download
import time

print("Downloading Perch v2 ONNX model...")
t0 = time.time()
onnx_path = hf_hub_download(
    repo_id="justinchuby/Perch-onnx",
    filename="perch_v2.onnx",
)
print(f"Downloaded in {time.time()-t0:.1f}s: {onnx_path}")

In [ ]:
# Cell 3 — Load ONNX model and check providers
import onnxruntime as ort
import numpy as np

print("Available providers:", ort.get_available_providers())

# GPU優先、なければCPU
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(onnx_path, providers=providers)
print("Active provider:", session.get_providers())

# 入出力の確認
for inp in session.get_inputs():
    print(f"Input: {inp.name}, shape={inp.shape}, dtype={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name}, shape={out.shape}, dtype={out.type}")

In [ ]:
# Cell 4 — Warm-up + single inference speed test
import time

SR = 32000
DURATION = 5  # seconds
dummy = np.random.randn(1, SR * DURATION).astype(np.float32)

# Warm-up (3 runs)
for _ in range(3):
    session.run(None, {"inputs": dummy})

# Single inference
t0 = time.time()
outputs = session.run(None, {"inputs": dummy})
t1 = time.time()
print(f"Single inference: {(t1-t0)*1000:.1f} ms")

# Output shapes
output_names = [o.name for o in session.get_outputs()]
for name, arr in zip(output_names, outputs):
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

In [ ]:
# Cell 5 — Batch inference speed test
import time

SR = 32000
DURATION = 5

results = []
for batch_size in [1, 4, 8, 12, 16, 32, 64]:
    dummy = np.random.randn(batch_size, SR * DURATION).astype(np.float32)
    
    # Warm-up
    try:
        session.run(None, {"inputs": dummy})
    except Exception as e:
        print(f"Batch size {batch_size}: FAILED - {e}")
        continue
    
    # Measure (5 runs average)
    times = []
    for _ in range(5):
        t0 = time.time()
        session.run(None, {"inputs": dummy})
        times.append(time.time() - t0)
    
    avg_ms = np.mean(times) * 1000
    per_sample_ms = avg_ms / batch_size
    results.append((batch_size, avg_ms, per_sample_ms))
    print(f"Batch {batch_size:3d}: {avg_ms:8.1f} ms total, {per_sample_ms:6.1f} ms/sample")

# 全 train_soundscapes の見積もり
print("\n--- 全 train_soundscapes 処理時間の見積もり ---")
TOTAL_FILES = 10592
WINDOWS_PER_FILE = 12
TOTAL_WINDOWS = TOTAL_FILES * WINDOWS_PER_FILE
print(f"Total windows: {TOTAL_WINDOWS:,}")

for batch_size, _, per_sample_ms in results:
    total_sec = TOTAL_WINDOWS * per_sample_ms / 1000
    total_min = total_sec / 60
    total_hr = total_min / 60
    print(f"  Batch {batch_size:3d}: {total_min:6.1f} min ({total_hr:.2f} hr)")

In [ ]:
# Cell 6 — Test with real audio files
import librosa
from pathlib import Path
import time

SR = 32000
DURATION = 5
SAMPLES = SR * DURATION  # 160000

# train_soundscapes のパス
SS_DIR = Path("/kaggle/input/birdclef-2026/train_soundscapes")
if not SS_DIR.exists():
    SS_DIR = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")

# 最初の10ファイルでテスト
test_files = sorted(SS_DIR.glob("*.ogg"))[:10]
print(f"Testing with {len(test_files)} files")

all_embeddings = []
all_logits = []
t_start = time.time()

for fpath in test_files:
    # Load and split into 5-second windows
    audio, _ = librosa.load(fpath, sr=SR, mono=True)
    n_windows = len(audio) // SAMPLES
    windows = []
    for i in range(n_windows):
        chunk = audio[i * SAMPLES : (i + 1) * SAMPLES]
        if len(chunk) == SAMPLES:
            windows.append(chunk)
    
    if not windows:
        continue
    
    # Batch inference (all windows of one file at once)
    batch = np.stack(windows).astype(np.float32)
    outputs = session.run(None, {"inputs": batch})
    
    # Find embedding and label outputs
    output_names = [o.name for o in session.get_outputs()]
    for name, arr in zip(output_names, outputs):
        if name == "embedding":
            all_embeddings.append(arr)
        elif name == "label":
            all_logits.append(arr)

t_elapsed = time.time() - t_start
n_windows_total = sum(e.shape[0] for e in all_embeddings)

print(f"\nProcessed {len(test_files)} files ({n_windows_total} windows) in {t_elapsed:.2f}s")
print(f"  Per file: {t_elapsed/len(test_files)*1000:.0f} ms")
print(f"  Per window: {t_elapsed/n_windows_total*1000:.1f} ms")
print(f"\nEmbedding shape: {all_embeddings[0].shape}")
print(f"Logits shape: {all_logits[0].shape}")

# 全ファイル見積もり
est_total = t_elapsed / len(test_files) * 10592
print(f"\n--- 全 10,592 ファイルの見積もり ---")
print(f"  {est_total/60:.1f} min ({est_total/3600:.2f} hr)")

In [ ]:
# Cell 7 — Compare with TF CPU (optional, comment out if TF not available)
# This cell compares ONNX GPU vs TF CPU to quantify the speedup

try:
    import tensorflow as tf
    
    # Load TF model
    MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
    if not MODEL_DIR.exists():
        MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1")
    
    if MODEL_DIR.exists():
        tf_model = tf.saved_model.load(MODEL_DIR)
        serve = tf_model.signatures['serving_default']
        input_key = list(serve.structured_input_signature[1].keys())[0]
        
        # Single file test with TF
        audio, _ = librosa.load(test_files[0], sr=SR, mono=True)
        chunk = audio[:SAMPLES].astype(np.float32)
        
        # Warm-up
        with tf.device('/CPU:0'):
            serve(**{input_key: tf.constant(chunk[np.newaxis, :])})
        
        # Measure TF CPU
        t0 = time.time()
        with tf.device('/CPU:0'):
            tf_out = serve(**{input_key: tf.constant(chunk[np.newaxis, :])})
        tf_time = time.time() - t0
        
        # Measure ONNX
        t0 = time.time()
        onnx_out = session.run(None, {"inputs": chunk[np.newaxis, :]})
        onnx_time = time.time() - t0
        
        print(f"TF CPU:   {tf_time*1000:.1f} ms")
        print(f"ONNX GPU: {onnx_time*1000:.1f} ms")
        print(f"Speedup:  {tf_time/onnx_time:.1f}x")
    else:
        print("TF model not found, skipping comparison")
except ImportError:
    print("TensorFlow not installed, skipping comparison")